# Step 1: Retrieve taxi and weather data from 2016 to 2019

In [1]:
# Running this code will query a table in BigQuery and download
# the results to a Pandas DataFrame named `results`.
# Learn more here: https://cloud.google.com/bigquery/docs/visualize-jupyter

%%bigquery --project project-ee65fd96-054c-4551-813

SELECT *
 FROM `bigquery-public-data.noaa_gsod.stations`
WHERE country = 'US' AND state = 'NY' AND LOWER(name) LIKE '%central park%'
LIMIT 100

Query is running:   0%|          |

Downloading:   0%|          |

,usaf,wban,name,country,state,call,lat,lon,elev,begin,end
0,725033,94728,NYC CENTRAL PARK,US,NY,KNYC,40.779,-73.969,+0047.5,19431201,20100817
1,725060,94728,NEW YORK CENTRAL PARK,US,NY,KNYC,40.779,-73.969,+0047.5,20100817,20120430
2,999999,94728,NEW YORK CENTRAL PARK,US,NY,KNYC,40.779,-73.969,+0047.5,19650101,20120731
3,725053,94728,CENTRAL PARK,US,NY,KNYC,40.779,-73.969,+0042.7,20050101,20210921


In [2]:
%%bigquery --project project-ee65fd96-054c-4551-813

CREATE OR REPLACE TABLE table.weather_raw AS (
  SELECT CAST(year AS INT) AS `year`, CAST(mo AS INT) AS `month`, CAST(da AS INT) AS `day`,
         temp, CAST(wdsp AS FLOAT64) AS wdsp, prcp, sndp
  FROM `bigquery-public-data.noaa_gsod.gsod20*`
  WHERE wban = '94728' and _TABLE_SUFFIX BETWEEN '16' AND '19'
)

Query is running:   0%|          |

""


In [3]:
%%bigquery --project project-ee65fd96-054c-4551-813

CREATE OR REPLACE TABLE table.taxi_raw AS (
  SELECT DATE_TRUNC(`pickup_datetime`, DAY) AS date, COUNT(*) AS pickup_count
  FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_20*`
  WHERE _TABLE_SUFFIX BETWEEN '16' AND '19' AND EXTRACT(YEAR FROM pickup_datetime) BETWEEN 2016 AND 2019
  GROUP BY DATE_TRUNC(`pickup_datetime`, DAY)
)

Query is running:   0%|          |

""


# Step 2: First insight

In [4]:
# Check distinct day per year in both table
%%bigquery --project project-ee65fd96-054c-4551-813

WITH taxi_day_count AS (
  SELECT EXTRACT(YEAR FROM date) AS year,
         COUNT(DISTINCT DATE_TRUNC(date, DAY)) AS day_count
  FROM table.taxi_raw
  GROUP BY EXTRACT(YEAR FROM date)
),
weather_day_count AS (
  SELECT year,
         COUNT(DISTINCT DATE(`year`, `month`, `day`)) AS day_count
  FROM table.weather_raw
  GROUP BY year
)

SELECT COALESCE(t.year, w.year) AS year,
       t.day_count AS taxi_day_count,
       w.day_count AS weather_day_count
FROM taxi_day_count AS t
FULL OUTER JOIN weather_day_count AS w
  ON t.year = w.year
ORDER BY t.year DESC;

Query is running:   0%|          |

Downloading:   0%|          |

,year,taxi_day_count,weather_day_count
0,2019,365,365
1,2018,365,365
2,2017,365,365
3,2016,366,366


The GSOD weather dataset specifies that missing values are represented not as NULL values but instead as floats or strings such as 999.9. These values need to be replaced otherwise they will be interpreted incorrectly


In [5]:

%%bigquery --project project-ee65fd96-054c-4551-813

CREATE OR REPLACE VIEW view.weather_clean AS (
  SELECT
    `year`,
    `month`,
    `day`,
    NULLIF(temp, 9999.9) AS temp,
    NULLIF(wdsp, 999.9) AS wdsp,
    NULLIF(prcp, 99.99) AS prcp,
    NULLIF(sndp, 999.9) AS sndp
  FROM table.weather_raw
)

Query is running:   0%|          |

""


In [6]:
# check null values in taxi table
%%bigquery --project project-ee65fd96-054c-4551-813

SELECT COUNTIF(`year` IS null) AS year_null,
       COUNTIF(`month` IS null) AS month_null,
       COUNTIF(`day` IS null) AS day_null,
       COUNTIF(`temp`IS null) AS temp_null,
       COUNTIF(wdsp IS null) AS wdsp_null,
       COUNTIF(prcp IS null) AS prcp_null,
       COUNTIF(sndp IS null) AS sndp_null
FROM view.weather_clean;

Query is running:   0%|          |

Downloading:   0%|          |

,year_null,month_null,day_null,temp_null,wdsp_null,prcp_null,sndp_null
0,0,0,0,0,161,0,1397


In [7]:
%%bigquery --project project-ee65fd96-054c-4551-813

SELECT

  MIN(temp) AS temp_min,
  MAX(temp) AS temp_max,

  MIN(wdsp) AS wdsp_min,
  MAX(wdsp) AS wdsp_max,

  MIN(prcp) AS prcp_min,
  MAX(prcp) AS prcp_max,

  MIN(sndp) AS sndp_min,
  MAX(sndp) AS sndp_max

FROM view.weather_clean;

Query is running:   0%|          |

Downloading:   0%|          |

,temp_min,temp_max,wdsp_min,wdsp_max,prcp_min,prcp_max,sndp_min,sndp_max
0,6.9,88.7,0.0,15.5,0.0,3.18,1.2,22.0


# Step 3: Data cleaning
The sndp feature is missing even more values, but the documentation of the dataset specifies that the weather station will often put *999.9* when there is no snow on the ground instead of *0*. For this reason, all missing values in the sndp feature will be imputed with 0.

In [8]:
%%bigquery --project project-ee65fd96-054c-4551-813
CREATE OR REPLACE VIEW view.weather_clean_tmp AS (
  SELECT
    COALESCE(sndp, 0) AS sndp,
    * EXCEPT(sndp)
  FROM view.weather_clean
  ORDER BY `year`, `month`, `day`
);

Query is running:   0%|          |

""


In [9]:
from google.cloud import bigquery
import numpy as np
client = bigquery.Client(project='project-ee65fd96-054c-4551-813')

df = client.query("SELECT * FROM view.weather_clean_tmp").to_dataframe()
df.interpolate(method="linear", inplace = True)
display(df[df['wdsp'].isna()])


table_id = "project-ee65fd96-054c-4551-813.table.weather_clean"

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE"
)

client.load_table_from_dataframe(
    df,
    table_id,
    job_config=job_config
).result()

,sndp,year,month,day,temp,wdsp,prcp


LoadJob<project=project-ee65fd96-054c-4551-813, location=US, id=e9554fcd-dd68-468d-b8cd-d68daffa5e82>

# Step 4: Feature engineering

In [17]:
# taxi count per day + day of week
%%bigquery --project project-ee65fd96-054c-4551-813
CREATE OR REPLACE TABLE `table.taxi_weather_clean` AS (
  WITH taxi_count_per_day AS (
    SELECT date,
          EXTRACT(DAY FROM date) AS `day`,
          EXTRACT(DAYOFWEEK FROM date) AS `day_of_week`,
          DATE_DIFF(date, DATE_TRUNC(date, YEAR), DAY) + 1 AS day_of_year,
          EXTRACT(MONTH FROM date) AS `month`,
          EXTRACT(YEAR FROM date) AS `year`,
          COALESCE(pickup_count, 0) AS pickup_count
    FROM table.taxi_raw
  )

  SELECT *
  FROM taxi_count_per_day t
  JOIN table.weather_clean AS w
    USING (day, month, year)
  ORDER BY date
);


Query is running:   0%|          |

""


In [16]:
%%bigquery --project project-ee65fd96-054c-4551-813
SELECT *
FROM table.taxi_weather_clean
LIMIT 100;

Query is running:   0%|          |

Downloading:   0%|          |

,day,month,year,date,day_of_week,day_of_year,pickup_count,sndp,temp,wdsp,prcp
0,1,1,2016,2016-01-01 00:00:00+00:00,6,1,345036,0.0,40.4,6.5,0.03
1,2,1,2016,2016-01-02 00:00:00+00:00,7,2,312830,0.0,35.8,6.0,0.00
2,3,1,2016,2016-01-03 00:00:00+00:00,1,3,302878,0.0,38.4,6.4,0.00
3,4,1,2016,2016-01-04 00:00:00+00:00,2,4,316008,0.0,30.8,7.0,0.00
4,5,1,2016,2016-01-05 00:00:00+00:00,3,5,343128,0.0,18.4,6.2,0.00
...,...,...,...,...,...,...,...,...,...,...,...
95,5,4,2016,2016-04-05 00:00:00+00:00,3,96,399543,0.0,33.0,7.9,0.46
96,6,4,2016,2016-04-06 00:00:00+00:00,4,97,400644,0.0,37.9,4.9,0.00
97,7,4,2016,2016-04-07 00:00:00+00:00,5,98,409439,0.0,52.4,5.8,0.00
98,8,4,2016,2016-04-08 00:00:00+00:00,6,99,440190,0.0,47.3,6.7,0.09
